### Structured Output


Models can ve requested to provide their response in a fromat matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. Langchain supports multiple schema types and methods for enforcing structured output.

### Pydantic

Pydantic models provide the richest feature set with field validation, description and nested structures.

In [6]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F6EE1EA810>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F6EE2D5DC0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [7]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str=Field(description="The title of the movie")
    year: str=Field(description="The year movie was released")
    director: str=Field(description="The director of the movie")
    rating: str=Field(description="The movie rating out of 10")
    


In [8]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F6EE1EA810>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F6EE2D5DC0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year movie was released', 'type': 'string'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movie rating out of 10'

In [9]:
model.invoke("Provide the details about the movie Inception")

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Inception. Let me start by recalling what I know about it. First, the director is Christopher Nolan, right? He\'s known for complex narratives and mind-bending concepts. The main actor is Leonardo DiCaprio, who plays the lead character, Dom Cobb. The movie is about dreams within dreams, something like that. There\'s a concept where they enter someone\'s subconscious to plant an idea, called "inception." \n\nI remember there\'s a device called the "kick," which is used to wake someone up from a dream. The story involves layers of dreams, each more dangerous as you go deeper. There\'s also a theme about reality versus dreams, and the character\'s personal journey involving his wife, Mal (played by Ellen Page). Wait, Ellen Page is in it too. Also, other actors like Joseph Gordon-Levitt, who does a lot of action scenes, maybe even a fight in a rotating hallway. Tom Hardy is another key actor, playing a character

In [10]:
model_with_structure.invoke("Provide the details about the movie Inception")

Movie(title='Inception', year='2010', director='Christopher Nolan', rating='8.8')

### Message output alongside Parsed Structure

In [11]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str=Field(description="The title of the movie")
    year: str=Field(description="The year movie was released")
    director: str=Field(description="The director of the movie")
    rating: str=Field(description="The movie rating out of 10")
    
model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide the details about the movie Inception")

response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for details about the movie Inception. Let me check what tools I have available. There's a Movie function that requires title, year, director, and rating. I need to fill those parameters. I know Inception was directed by Christopher Nolan. The release year was 2010. The rating is probably around 8.8 on IMDb. Let me confirm those details. Yep, that's correct. So I'll structure the tool call with those parameters.\n", 'tool_calls': [{'id': 'cparjd6x2', 'function': {'arguments': '{"director":"Christopher Nolan","rating":"8.8","title":"Inception","year":"2010"}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 149, 'prompt_tokens': 225, 'total_tokens': 374, 'completion_time': 0.260010763, 'completion_tokens_details': {'reasoning_tokens': 101}, 'prompt_time': 0.009380684, 'prompt_tokens_details': None, 'queue_time': 0.160113725, 'total_time': 0.269

### Nested Structure


In [12]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name: str
    role:str
    
class MovieDetails(BaseModel):
    title:str
    year: int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None, description="Budget in million USD")


In [13]:
model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide the details about the movie Inception")

response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames')], genres=['Science Fiction', 'Action'], budget=160.0)

### TypeDict


TypeDict provides a simpler alternative using Pythons built-in typing, ideal when you dont need runtime validation

In [14]:
from typing_extensions import TypedDict, Annotated

In [15]:
class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie rating out of 10"]

In [16]:
model_with_typeDict = model.with_structured_output(MovieDict)

response = model_with_typeDict.invoke("Provide the details about the movie Inception")

response

{'director': 'Christopher Nolan',
 'rating': 8.8,
 'title': 'Inception',
 'year': 2010}

In [18]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}